In [18]:
from typing import List, TypedDict, Literal
from pydantic import BaseModel, Field
import time

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

True

In [19]:
docs = (
    PyPDFLoader("../documents/Company_Policies.pdf").load() + 
    PyPDFLoader("../documents/Company_Profile.pdf").load() + 
    PyPDFLoader("../documents/Product_and_Pricing.pdf").load() 
)

In [20]:
chunks = RecursiveCharacterTextSplitter(
    chunk_size=600, 
    chunk_overlap=150, 
).split_documents(docs)

In [21]:
embeddings = OllamaEmbeddings(model='bge-m3')
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="../vector-database/selfrag3"
)

In [22]:
llm = ChatOllama(model="llama3.2", temperature=0)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})



In [23]:
# Graph state 
class State(TypedDict): 
    question: str
    need_retrieval: bool

    docs: List[Document]
    relevant_docs: List[Document]

    context: str
    answer: str

    #post generation verification
    issup: Literal["fully_supported", "partially_supported", "np_support"]
    evidence: List[str]

In [24]:
class RetrieveDecision(BaseModel):
    should_retrieve: bool = Field(
        ...,
        description="True if external documents are needed to answer reliably, else False."
    )

decide_retrieval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You decide whether retrieval is needed.\n"
            "We have internal documents about a company called NexaAI: "
            "company policies, company profile (CEO, team, history), "
            "and products and pricing.\n"
            "Return JSON that matches this schema:\n"
            "{{'should_retrieve': boolean}}\n\n"
            "Guidelines:\n"
            "- should_retrieve=True for ANY question about NexaAI, its people, "
            "policies, products, or prices.\n"
            "- should_retrieve=False ONLY for general knowledge like "
            "'what is machine learning'.\n"
            "- If unsure, choose True."
        ),
        ("human", "Question: {question}"),
    ]
)


# IMPORTANT: no `.content` for structured output
should_retrieve_llm = llm.with_structured_output(RetrieveDecision)

def decide_retrieval(state: "State"):
    decision: RetrieveDecision = should_retrieve_llm.invoke(
        decide_retrieval_prompt.format_messages(question=state["question"])
    )
    return {"need_retrieval": decision.should_retrieve}

In [25]:

direct_generation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer the question using only your general knowledge.\n"
            "Do NOT assume access to external documents.\n"
            "If you are unsure or the answer requires specific sources, say:\n"
            "'I don't know based on my general knowledge.'"
        ),
        ("human", "{question}"),
    ]
)


def generate_direct(state: State):
    out = llm.invoke(
        direct_generation_prompt.format_messages(
            question=state["question"]
        )
    )
    return {
        "answer": out.content
    }

In [26]:
def retrieve(state: State):
    return {"docs": retriever.invoke(state["question"])}

In [27]:
class RelevanceDecision(BaseModel): 
    is_relevant: bool = Field(
        ..., 
        description="True if the document helps answer the question, else False"
    )

In [28]:
is_relevant_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system", 
            "You are judging document relevance. \n"
            "Return JSON that matches this schema: \n"
            "{{'is_relevant': boolean}}\n\n"
            "A document is relevant if it contains information useful for answering the question."

        ), 
        (
            "human", 
            "Question:\n {question}\n\n Document: \n{document}"
        )
    ]
)

relevance_llm = llm.with_structured_output(RelevanceDecision)

def is_relevant(state: State): 
    relevant_docs: List[Document] = []
    for doc in state['docs']: 
        decision: RelevanceDecision = relevance_llm.invoke(
            is_relevant_prompt.format_messages(
                question=state['question'], 
                document = doc.page_content
            )
        )
        if decision.is_relevant: 
            relevant_docs.append(doc)
    return {"relevant_docs": relevant_docs}

In [29]:
rag_generation_promt = ChatPromptTemplate(
    [
        (
            "system", 
            "You are a business RAG assistant.\n"
            "Answer the user's question using ONLY the provided context.\n"
            "If the context does not contain enough information.\n"
            "No relevant document found.\n"
            "Do not use outside knowledge.\n" 
        ), 
        (
            "human", 
            "Question: \n {question}\n\n"
            "Context: \n{context}\n"
        )
    ]
)

def generate_from_context(state: State): 
    # Stuff relevant docs into one block
    context = "\n\n---\n\n".join(
        [d.page_content for d in state.get("relevant_docs", [])]
    ).strip()

    if not context: 
        return {"answer": "No relevant document found.", "context": ""}

    out = llm.invoke(
        rag_generation_promt.format_messages(
            question=state['question'], 
            context=context
        )
    )
    return {"answer": out.content, "context": context}

In [42]:
def router_after_relevance(state: State) -> Literal['generate_from_context', 'no_relevant_docs']: 
    if state.get("relevant_docs") and len(state['relevant_docs']) > 0: 
        return "generate_from_context"
    return "no_relevant_docs"

def no_answer_found(state: State):
    return {"answer": "No answer found.", "context": ""}
def route_after_decide(state: State) -> Literal["generate_direct", "retrieve"]:
    if state['need_retrieval']:
        return "retrieve"
    return "generate_direct"


In [43]:
#IsSup verify and revise loop
class IsSupDecision(BaseModel): 
    issup: Literal['fully_supported', "partially_supported", "no_support"]
    evidence: List[str] = Field(default_factory=list)

issue_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system", 
            ""
        )
    ]
)

issup_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are verifying whether the ANSWER is supported by the CONTEXT.\n"
            "Return JSON with keys: issup, evidence.\n"
            "issup must be one of: fully_supported, partially_supported, no_support.\n\n"
            "How to decide issup:\n"
            "- fully_supported:\n"
            "  Every meaningful claim is explicitly supported by CONTEXT, and the ANSWER does NOT introduce\n"
            "  any qualitative/interpretive words that are not present in CONTEXT.\n"
            "  (Examples of disallowed words unless present in CONTEXT: culture, generous, robust, designed to,\n"
            "  supports professional development, best-in-class, employee-first, etc.)\n\n"
            "- partially_supported:\n"
            "  The core facts are supported, BUT the ANSWER includes ANY abstraction, interpretation, or qualitative\n"
            "  phrasing not explicitly stated in CONTEXT (e.g., calling policies 'culture', saying leave is 'generous',\n"
            "  or inferring outcomes like 'supports professional development').\n\n"
            "- no_support:\n"
            "  The key claims are not supported by CONTEXT.\n\n"
            "Rules:\n"
            "- Be strict: if you see ANY unsupported qualitative/interpretive phrasing, choose partially_supported.\n"
            "- If the answer is mostly unrelated to the question or unsupported, choose no_support.\n"
            "- Evidence: include up to 3 short direct quotes from CONTEXT that support the supported parts.\n"
            "- Do not use outside knowledge."
        ),
        (
            "human",
            "Question:\n{question}\n\n"
            "Answer:\n{answer}\n\n"
            "Context:\n{context}\n"
        ),
    ]
)



issup_llm = llm.with_structured_output(IsSupDecision)


In [44]:
def is_sup(state: State):
    decision: IsSupDecision = issup_llm.invoke(
        issup_prompt.format_messages(
            question=state['question'],
            answer=state.get('answer', ""), 
            context= state.get("context", "")
        )
    )
    return {"issup": decision.issup, "evidence": decision.evidence}

In [45]:
g = StateGraph(State)

g.add_node("decide_retrieval", decide_retrieval)
g.add_node("generate_direct", generate_direct)
g.add_node("retrieve", retrieve)

g.add_node("is_relevant", is_relevant)
g.add_node("generate_from_context", generate_from_context)
g.add_node("no_answer_found", no_answer_found)

g.add_node("is_sup", is_sup)

g.add_edge(START, "decide_retrieval")

g.add_conditional_edges(
    "decide_retrieval",
    route_after_decide,
    {"generate_direct": "generate_direct", "retrieve": "retrieve"},
)

g.add_edge("generate_direct", END)

g.add_edge("retrieve", "is_relevant")

g.add_conditional_edges(
    "is_relevant",
    router_after_relevance,
    {"generate_from_context": "generate_from_context", "no_answer_found": "no_answer_found"},
)

# If no answer found, end
g.add_edge("no_answer_found", END)

# If generated from context, verify with IsSUP loop
g.add_edge("generate_from_context", "is_sup")
g.add_edge("is_sup", END)

app = g.compile()
app


ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`